# CS781 Phase 5 — Experiment 6: Knowledge Distillation
## Model4 Partial-FT Teacher → Model2 Student (Two-Stage KD + True-Label Consolidation, FAST-SCORE)

**Goal:** maximize Phase-5 Legal Statute Identification Macro-F1 while keeping this experiment methodologically distinct as **Knowledge Distillation**.

### Teacher
The teacher is the trained **Model4 Partial / Layer-wise Fine-Tuning** classifier from Experiment 1.

- honest heldout teacher: `best_partialft_lsi_v3.pt` — trained only on TRAIN
- final refit teacher: `refit_model_selected_weights.pt` — trained on TRAIN + DEV

### Student
The student starts from the exact **Phase-1 Model2 decoder-only checkpoint**.

Model2:
- 13 decoder blocks
- d_model = 640
- 10 attention heads
- FFN = 2560
- sinusoidal positional encoding
- pre-LN + GELU
- tied token/output embeddings

### Two-stage training

#### Stage A — teacher-guided distillation
The student learns:
1. the teacher's **soft 100-label distribution** using temperature-scaled Bernoulli distillation;
2. a smaller amount of the true hard-label BCE objective;
3. a weak **representation distillation** loss over the four document views.

Because both models use 640-dimensional hidden states and 384-dimensional projected view representations, the teacher's view-level representations can guide the student in addition to the final logits.

#### Stage B — true-label consolidation
Teacher losses are turned **completely off**.

The student then trains only on the real statute labels with:
- softened class-balanced BCE
- asymmetric multilabel loss for rare-label recall

This is intentional: the teacher gives the student a strong initialization/decision structure, then the ground truth gets the final say.

### Leakage-safe design

- Heldout student:
  - KD teacher = TRAIN-only `best_partialft_lsi_v3.pt`
  - train = TRAIN
  - select checkpoint/thresholds on clean DEV
- Final student refit:
  - KD teacher = TRAIN+DEV `refit_model_selected_weights.pt`
  - train = TRAIN+DEV
  - **no** post-refit threshold tuning on DEV
  - reuse heldout-selected threshold vector

### Core outputs

1. KD student HELDOUT ZIP
2. KD student TRAIN+DEV REFIT ZIP
3. HELDOUT + REFIT student probability ensemble ZIP
4. optional teacher + student ensemble ZIPs when Experiment-1 probability files are attached

The ensemble variants are submission variants, not separate adaptation methods.


## V3 — FAST-SCORE version

This version keeps the useful **runtime optimizations** from V2 but removes the changes that merely make training longer.

### Kept because they reduce runtime without changing the KD objective
- `FAST_CAUSAL_NO_MASK=True` for the right-padded causal student path, enabling the fast SDPA causal kernel.
- Student micro-batch **4** with grad accumulation **4**: effective document batch remains **16**, but GPU utilization is much better than micro-batch 1.
- Larger DEV/teacher inference batches.
- Fused multi-tensor EMA updates.
- No per-step CPU synchronization for running loss statistics.
- Shared training loop for Stage A, Stage B, and refit.
- Stage-A DEV diagnostic only after its final epoch.
- Parallelized token-cache construction and better worker cleanup.
- Heldout resources are released before the TRAIN+DEV refit.
- Optional teacher/student **4-way probability blend** if the Experiment-1 `.npz` files are attached; this adds negligible runtime.

### Deliberately NOT kept
V2 increased:
- KD epochs: **2 → 3**
- true-label epochs: **2 → 3**

Those changes substantially increase runtime. V3 returns to:
- **2 KD epochs**
- **2 supervised epochs**

The core learning strategy is unchanged:

1. **Stage A:** Model2 learns from Model4 soft logits + a smaller hard-label term + weak view-representation matching.
2. **Stage B:** teacher loss becomes zero and Model2 learns only from the real statute labels.
3. Clean DEV selects the honest heldout student and thresholds.
4. Optional TRAIN+DEV refit starts fresh from Phase-1 Model2 and uses the TRAIN+DEV Model4 teacher.


## Kaggle inputs

Attach these assets:

**Student**
- Phase-1 Model2 `dec_full_best*.pt`
- exact Model2 `tokenizer.json`

**Teacher**
- `best_partialft_lsi_v3.pt`
- `refit_model_selected_weights.pt`
- exact Model4 `tokenizer.json`

Optional but useful:
- Experiment-1 `dev_official_probs.npz`
- Experiment-1 `heldout_test_probs.npz`
- Experiment-1 `refit_test_probs.npz`

If auto-detection selects the wrong tokenizer/checkpoint, use the override variables in Cell 1.


In [1]:
# CELL 1 — imports + exact Model2 / KD configuration
import os, re, gc, json, math, time, random, hashlib, zipfile, shutil, contextlib, subprocess, sys, importlib.util, heapq
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, accuracy_score
from huggingface_hub import hf_hub_download, login

SEED = 6789
NOTEBOOK_VERSION = "exp6_model4_to_model2_kd_twostage_fastscore_v3"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/phase5_exp6_kd_model4_to_model2_scoremax")
CACHE_DIR = ROOT / "cache"
TEACHER_CACHE = ROOT / "teacher_cache"
ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_CACHE.mkdir(parents=True, exist_ok=True)

# ---------------- exact Model2 ----------------
M2_VOCAB_SIZE = 24_576
M2_D_MODEL = 640
M2_N_LAYERS = 13
M2_N_HEADS = 10
M2_FFN_DIM = 2560
M2_DROPOUT = 0.0
M2_MAX_POSITIONS = 2048

# ---------------- exact Model4 teacher ----------------
M4_D_MODEL = 640
M4_N_LAYERS = 17
M4_N_HEADS = 10
M4_N_KV_HEADS = 2
M4_FFN_DIM = 2048
M4_DROPOUT = 0.0
M4_MAX_POSITIONS = 2560
M4_ROPE_BASE = 10_000.0
M4_RMS_EPS = 1e-5
M4_FROZEN_BLOCKS = 12

# ---------------- Phase 5 ----------------
N_LABELS = 100
N_CHUNKS = 4
CHUNK_LEN = 384
CHAR_WINDOW = 6000
VIEW_NAMES = ["head", "middle", "statute_evidence", "tail"]

# ---------------- student architecture/training ----------------
# Score-first: entire Model2 transformer stack is trainable.
# Token embedding is frozen to save memory and reduce catastrophic drift.
FREEZE_TOKEN_EMBEDDING = True
USE_GRAD_CHECKPOINTING = True
FAST_CAUSAL_NO_MASK = True   # exact for right-padded causal attention; enables flash/SDPA is_causal path

MICRO_BATCH_SIZE = 4   # was 1; effective batch unchanged (4 x 4 = 16)
DEV_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4    # was 16
NUM_WORKERS = 4
USE_AMP = True
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EMA_DECAY = 0.9985

# Stage A: teacher-guided.
KD_EPOCHS = 2
KD_TEMPERATURE = 2.5
KD_SOFT_WEIGHT = 0.72
KD_HARD_WEIGHT = 0.23
KD_FEATURE_WEIGHT = 0.05

KD_TOP_BLOCK_LR = 4.0e-5
KD_LAYER_DECAY = 0.90
KD_HEAD_LR = 4.0e-4

# Stage B: true-label consolidation only.
SUP_EPOCHS = 2
SUP_TOP_BLOCK_LR = 2.0e-5
SUP_LAYER_DECAY = 0.90
SUP_HEAD_LR = 2.0e-4

# Hard-label objective.
POS_WEIGHT_CLIP = 10.0
SUP_BCE_WEIGHT = 0.80
SUP_ASL_WEIGHT = 0.20
ASL_GAMMA_NEG = 4.0
ASL_GAMMA_POS = 1.0
ASL_CLIP = 0.05

# Threshold tuning.
THR_GRID = np.round(np.arange(0.01, 0.8501, 0.01), 3)
SHRINK_TAUS = [15.0, 25.0, 50.0, 100.0, 200.0, 400.0]
CV_REPEATS = 3

# Final full-data student refit.
RUN_REFIT = True
REFIT_SEED = SEED + 999
REFIT_ENSEMBLE_WEIGHT = 0.50

# Input overrides.
MODEL2_CKPT_OVERRIDE = None
MODEL2_TOKENIZER_OVERRIDE = None
TEACHER_HELDOUT_CKPT_OVERRIDE = None
TEACHER_REFIT_CKPT_OVERRIDE = None
MODEL4_TOKENIZER_OVERRIDE = None

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def norm_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def text_hash(s):
    return hashlib.sha256(norm_text(s).encode("utf-8")).hexdigest()

def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

print("ROOT:", ROOT)
print("Views:", VIEW_NAMES, "x", CHUNK_LEN)
print("KD weights:", KD_SOFT_WEIGHT, KD_HARD_WEIGHT, KD_FEATURE_WEIGHT)
print("KD temperature:", KD_TEMPERATURE)
print("Stage A epochs:", KD_EPOCHS, "| Stage B epochs:", SUP_EPOCHS)


Device: cuda
GPU: Tesla T4
ROOT: /kaggle/working/phase5_exp6_kd_model4_to_model2_scoremax
Views: ['head', 'middle', 'statute_evidence', 'tail'] x 384
KD weights: 0.72 0.23 0.05
KD temperature: 2.5
Stage A epochs: 2 | Stage B epochs: 2


In [2]:
# CELL 2 — locate student + teacher assets robustly
all_files = []
for base in ["/kaggle/input", "/kaggle/working"]:
    if os.path.exists(base):
        for root, _, files in os.walk(base):
            for f in files:
                all_files.append(os.path.join(root, f))

def score_path(p, words):
    s = p.lower()
    return sum(10 for w in words if w in s) - len(p)/100000.0

# Student Model2 Phase-1.
m2_ckpts = [
    p for p in all_files
    if p.lower().endswith(".pt")
    and "dec_full_best" in os.path.basename(p).lower()
]
assert m2_ckpts or MODEL2_CKPT_OVERRIDE, "Attach Model2 dec_full_best*.pt"
MODEL2_CKPT = (
    str(MODEL2_CKPT_OVERRIDE)
    if MODEL2_CKPT_OVERRIDE
    else sorted(
        m2_ckpts,
        key=lambda p: score_path(p, ["model2", "decoder-only", "phase1"]),
        reverse=True,
    )[0]
)

# Teacher checkpoints.
heldout_cands = [
    p for p in all_files
    if os.path.basename(p).lower() == "best_partialft_lsi_v3.pt"
]
refit_cands = [
    p for p in all_files
    if os.path.basename(p).lower() == "refit_model_selected_weights.pt"
]

assert heldout_cands or TEACHER_HELDOUT_CKPT_OVERRIDE, "Attach best_partialft_lsi_v3.pt"
TEACHER_HELDOUT_CKPT = (
    str(TEACHER_HELDOUT_CKPT_OVERRIDE)
    if TEACHER_HELDOUT_CKPT_OVERRIDE
    else heldout_cands[0]
)

if TEACHER_REFIT_CKPT_OVERRIDE:
    TEACHER_REFIT_CKPT = str(TEACHER_REFIT_CKPT_OVERRIDE)
elif refit_cands:
    TEACHER_REFIT_CKPT = refit_cands[0]
else:
    TEACHER_REFIT_CKPT = None

# Tokenizers: prefer tokenizer in same folder as each checkpoint.
tok_cands = [
    p for p in all_files
    if os.path.basename(p).lower() == "tokenizer.json"
]
assert tok_cands, "No tokenizer.json found."

def choose_tok(ckpt, override, hints):
    if override:
        return str(override)
    cdir = os.path.dirname(ckpt).lower()
    return sorted(
        tok_cands,
        key=lambda p: (
            100 if os.path.dirname(p).lower() == cdir else 0,
            score_path(p, hints),
        ),
        reverse=True,
    )[0]

MODEL2_TOKENIZER = choose_tok(
    MODEL2_CKPT,
    MODEL2_TOKENIZER_OVERRIDE,
    ["model2", "decoder-only", "phase1"],
)
MODEL4_TOKENIZER = choose_tok(
    TEACHER_HELDOUT_CKPT,
    MODEL4_TOKENIZER_OVERRIDE,
    ["model4", "partial", "phase2-model4", "phase2_model4"],
)

print("MODEL2_CKPT:", MODEL2_CKPT)
print("MODEL2_TOKENIZER:", MODEL2_TOKENIZER)
print("TEACHER_HELDOUT:", TEACHER_HELDOUT_CKPT)
print("TEACHER_REFIT:", TEACHER_REFIT_CKPT)
print("MODEL4_TOKENIZER:", MODEL4_TOKENIZER)

assert os.path.exists(MODEL2_CKPT)
assert os.path.exists(MODEL2_TOKENIZER)
assert os.path.exists(TEACHER_HELDOUT_CKPT)
assert os.path.exists(MODEL4_TOKENIZER)

print("Model2 SHA:", sha256_file(MODEL2_CKPT))
print("Teacher heldout SHA:", sha256_file(TEACHER_HELDOUT_CKPT))
if TEACHER_REFIT_CKPT:
    print("Teacher refit SHA:", sha256_file(TEACHER_REFIT_CKPT))


MODEL2_CKPT: /kaggle/input/datasets/aaryant23/phase2-model4/dec_full_best (8).pt
MODEL2_TOKENIZER: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
TEACHER_HELDOUT: /kaggle/input/datasets/aaryant23/phase2-model4/best_partialft_lsi_v3.pt
TEACHER_REFIT: /kaggle/input/datasets/aaryant23/phase2-model4/refit_model_selected_weights.pt
MODEL4_TOKENIZER: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Model2 SHA: 7685624ff48e4c2fbf213f9be4f626173e52676b007b9bc50dc44b0323d3cc15
Teacher heldout SHA: 94388658184d647a730efd219f45eeb3ac3ae178a7f53fdaf9e87edbdfd39ce2
Teacher refit SHA: 458a449abe02bc3ec99055d7cddd6cd8b90fc0da1bef61ad1fe506c2abdd37ef


In [3]:
# CELL 3 — exact LegalBPE class from supplied Phase-1 Model2 notebook
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

student_tokenizer = LegalBPE.load_json(MODEL2_TOKENIZER)
teacher_tokenizer = LegalBPE.load_json(MODEL4_TOKENIZER)

print("Student tokenizer vocab:", len(student_tokenizer))
print("Teacher tokenizer vocab:", len(teacher_tokenizer))
assert len(student_tokenizer) == M2_VOCAB_SIZE
assert len(teacher_tokenizer) == 24576


Tokenizer class ready.
Student tokenizer vocab: 24576
Teacher tokenizer vocab: 24576


In [4]:
# CELL 4 — official Phase-5 data
# CELL 4 — authenticate and download official Phase-5 data
from kaggle_secrets import UserSecretsClient
HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN and len(HF_TOKEN)>10, "HF_TOKEN secret missing."
login(token=HF_TOKEN,add_to_git_credential=False)

HF_REPO="Exploration-Lab/CS-781-Capstone"
FNS={
    "train":"Phase_5_LSI/Train/train.json",
    "dev":"Phase_5_LSI/Train/dev.json",
    "test":"Phase_5_LSI/Test/test.json",
}
paths={k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN) for k,v in FNS.items()}
DATA_SHA256={k:sha256_file(v) for k,v in paths.items()}
print("Dataset SHA256:")
for k,v in DATA_SHA256.items(): print(" ",k,v)

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])
print("raw rows:",len(train_raw),len(dev_raw),len(test_raw))
assert len(train_raw)==42750 and len(dev_raw)==10181 and len(test_raw)==13019

for name,recs,labeled in [("TRAIN",train_raw,True),("DEV",dev_raw,True),("TEST",test_raw,False)]:
    ids=[str(r["id"]) for r in recs]
    assert len(ids)==len(set(ids)),f"{name} duplicate IDs"
    for r in recs:
        assert isinstance(r.get("text"),str)
        if labeled:
            assert isinstance(r.get("labels"),list)
            # Dataset contains a few duplicate label entries; collapse duplicates as set semantics.
            r["labels"]=sorted(set(int(x) for x in r["labels"]))
            assert all(0<=x<N_LABELS for x in r["labels"])

assert not any("labels" in r for r in test_raw),"TEST labels unexpectedly public."
print("✅ schema / ID / label sanity PASS")


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Dataset SHA256:
  train f3601a7fce994616daf99fec251ca7936e98675a8fbf8af20a782cd55054c8a7
  dev ab2660ac23abf38e902d63ab6871df740dc38e3fd973f441745f42564dadbd15
  test c699d50bcc2f3e369955268d6195a80b23322ed91cb1fadc8d58b26ca8e9aa77
raw rows: 42750 10181 13019
✅ schema / ID / label sanity PASS


In [5]:
# CELL 5 — leakage-aware DEV + label statistics
train_hashes = {text_hash(r["text"]) for r in train_raw}
dev_hashes = {text_hash(r["text"]) for r in dev_raw}
test_hashes = {text_hash(r["text"]) for r in test_raw}

CLEAN_IDX = np.asarray(
    [i for i, r in enumerate(dev_raw) if text_hash(r["text"]) not in train_hashes],
    dtype=np.int64,
)
clean_dev = [dev_raw[i] for i in CLEAN_IDX]

print("unique normalized:", len(train_hashes), len(dev_hashes), len(test_hashes))
print("overlap train-dev:", len(train_hashes & dev_hashes))
print("overlap train-test:", len(train_hashes & test_hashes))
print("overlap dev-test:", len(dev_hashes & test_hashes))
print("official DEV:", len(dev_raw), "clean DEV:", len(clean_dev))

train_records = train_raw
test_records = test_raw

train_counts = np.zeros(N_LABELS, dtype=np.int64)
for r in train_records:
    train_counts[r["labels"]] += 1

raw_pw = (len(train_records) - train_counts) / train_counts
POS_WEIGHT = np.clip(
    np.sqrt(raw_pw),
    1.0,
    POS_WEIGHT_CLIP,
).astype(np.float32)

official_truth = [
    {"id": str(r["id"]), "gold": [int(x) for x in r["labels"]]}
    for r in dev_raw
]
clean_truth = [
    {"id": str(r["id"]), "gold": [int(x) for x in r["labels"]]}
    for r in clean_dev
]

print(
    "TRAIN counts min/median/max:",
    train_counts.min(),
    np.median(train_counts),
    train_counts.max(),
)
print(
    "pos_weight min/median/max:",
    POS_WEIGHT.min(),
    np.median(POS_WEIGHT),
    POS_WEIGHT.max(),
)


unique normalized: 42315 10118 12939
overlap train-dev: 81
overlap train-test: 109
overlap dev-test: 62
official DEV: 10181 clean DEV: 10063
TRAIN counts min/median/max: 272 679.0 11730
pos_weight min/median/max: 1.6261922 7.871644 10.0


In [6]:
# CELL 6 — shared four-view selector + separate student/teacher token caches
CUE_RE = re.compile(
    r"\b(section|sections|sec\.?|article|articles|act|acts|rule|rules|"
    r"provision|provisions|ipc|crpc|constitution)\b",
    re.I,
)

def _char_window(text, center, width=CHAR_WINDOW):
    if len(text) <= width:
        return text
    st = max(0, min(len(text) - width, int(center - width // 2)))
    return text[st:st + width]

def select_views(text):
    n = len(text)
    if n <= CHAR_WINDOW:
        return [text, text, text, text]

    head = text[:CHAR_WINDOW]
    tail = text[-CHAR_WINDOW:]
    mid = _char_window(text, n // 2)

    hits = [m.start() for m in CUE_RE.finditer(text)]
    if hits:
        best_i = best_j = 0
        j = 0
        for i in range(len(hits)):
            if j < i:
                j = i
            while j + 1 < len(hits) and hits[j + 1] - hits[i] <= CHAR_WINDOW:
                j += 1
            if j - i > best_j - best_i:
                best_i, best_j = i, j
        center = (hits[best_i] + hits[best_j]) // 2
        evid = _char_window(text, center)
    else:
        evid = mid

    return [head, mid, evid, tail]

def encode_view(tok, text, view_idx):
    ids = tok.encode(text)
    if not ids:
        ids = [tok.unk_token_id]

    if len(ids) > CHUNK_LEN:
        if view_idx == 0:
            ids = ids[:CHUNK_LEN]
        elif view_idx == 3:
            ids = ids[-CHUNK_LEN:]
        else:
            st = (len(ids) - CHUNK_LEN) // 2
            ids = ids[st:st + CHUNK_LEN]
    return ids

_WORK_TOK = None

def _encode_record(text):
    return [encode_view(_WORK_TOK, txt, c) for c, txt in enumerate(select_views(text))]

def cache_split(prefix, split_name, records, tok, tok_path):
    ids_path = CACHE_DIR / f"{prefix}_{split_name}_ids.npy"
    len_path = CACHE_DIR / f"{prefix}_{split_name}_lens.npy"
    meta_path = CACHE_DIR / f"{prefix}_{split_name}_meta.json"

    fingerprint = {
        "dataset_sha256": DATA_SHA256["dev" if split_name == "dev" else split_name],
        "tokenizer_sha256": sha256_file(tok_path),
        "n": len(records),
        "n_chunks": N_CHUNKS,
        "chunk_len": CHUNK_LEN,
        "char_window": CHAR_WINDOW,
        "ids_hash": hashlib.sha256(
            "|".join(str(r["id"]) for r in records).encode()
        ).hexdigest(),
    }

    if ids_path.exists() and len_path.exists() and meta_path.exists():
        if json.load(open(meta_path)) == fingerprint:
            print("✅ reuse cache:", prefix, split_name)
            return (
                np.load(ids_path, mmap_mode="r"),
                np.load(len_path, mmap_mode="r"),
            )

    arr = np.full(
        (len(records), N_CHUNKS, CHUNK_LEN),
        tok.pad_token_id,
        dtype=np.uint16,
    )
    lens = np.zeros(
        (len(records), N_CHUNKS),
        dtype=np.uint16,
    )

    global _WORK_TOK
    _WORK_TOK = tok
    texts = (r["text"] for r in records)
    pool = None
    try:
        import multiprocessing as mp
        pool = mp.get_context("fork").Pool(max(1, min(4, os.cpu_count() or 1)))
        it = pool.imap(_encode_record, texts, chunksize=32)
    except Exception as e:
        print("parallel tokenisation unavailable, falling back:", e)
        pool = None
        it = map(_encode_record, texts)

    t0 = time.time()
    for i, views in enumerate(it):
        for c, z in enumerate(views):
            L = min(len(z), CHUNK_LEN)
            arr[i, c, :L] = np.asarray(z[:L], dtype=np.uint16)
            lens[i, c] = L

        if (i + 1) % 2000 == 0 or i + 1 == len(records):
            print(
                f"{prefix}/{split_name}: {i+1}/{len(records)} "
                f"elapsed={(time.time()-t0)/60:.1f}m",
                flush=True,
            )
    if pool is not None:
        pool.close()
        pool.join()

    np.save(ids_path, arr)
    np.save(len_path, lens)
    json.dump(fingerprint, open(meta_path, "w"), indent=2)

    del arr, lens
    gc.collect()

    return (
        np.load(ids_path, mmap_mode="r"),
        np.load(len_path, mmap_mode="r"),
    )

# Student Model2 caches.
s_train_tok, s_train_lens = cache_split(
    "student", "train", train_records,
    student_tokenizer, MODEL2_TOKENIZER,
)
s_dev_tok, s_dev_lens = cache_split(
    "student", "dev", dev_raw,
    student_tokenizer, MODEL2_TOKENIZER,
)
s_test_tok, s_test_lens = cache_split(
    "student", "test", test_records,
    student_tokenizer, MODEL2_TOKENIZER,
)

# Teacher Model4 caches.
t_train_tok, t_train_lens = cache_split(
    "teacher", "train", train_records,
    teacher_tokenizer, MODEL4_TOKENIZER,
)
t_dev_tok, t_dev_lens = cache_split(
    "teacher", "dev", dev_raw,
    teacher_tokenizer, MODEL4_TOKENIZER,
)

print("student train:", s_train_tok.shape)
print("teacher train:", t_train_tok.shape)


student/train: 2000/42750 elapsed=0.1m
student/train: 4000/42750 elapsed=0.1m
student/train: 6000/42750 elapsed=0.2m
student/train: 8000/42750 elapsed=0.3m
student/train: 10000/42750 elapsed=0.3m
student/train: 12000/42750 elapsed=0.4m
student/train: 14000/42750 elapsed=0.5m
student/train: 16000/42750 elapsed=0.5m
student/train: 18000/42750 elapsed=0.6m
student/train: 20000/42750 elapsed=0.7m
student/train: 22000/42750 elapsed=0.7m
student/train: 24000/42750 elapsed=0.8m
student/train: 26000/42750 elapsed=0.8m
student/train: 28000/42750 elapsed=0.9m
student/train: 30000/42750 elapsed=0.9m
student/train: 32000/42750 elapsed=1.0m
student/train: 34000/42750 elapsed=1.1m
student/train: 36000/42750 elapsed=1.1m
student/train: 38000/42750 elapsed=1.2m
student/train: 40000/42750 elapsed=1.3m
student/train: 42000/42750 elapsed=1.3m
student/train: 42750/42750 elapsed=1.3m
student/dev: 2000/10181 elapsed=0.1m
student/dev: 4000/10181 elapsed=0.2m
student/dev: 6000/10181 elapsed=0.2m
student/dev: 

In [7]:
# CELL 7 — exact supplied Phase-1 Model2 architecture
# Cell 9 — ~80M Decoder-only causal Transformer with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        # Training / ordinary prefill: use SDPA's efficient causal path.
        # Left-padded batched prefill needs an explicit causal + key-valid mask.
        # Cached one-token decoding already has only past/current keys, so a key-valid mask is enough.
        attn_mask = None
        is_causal = False

        if key_valid is not None:
            key_mask = key_valid[:, None, None, :]  # (B,1,1,S), True = real key
            if past_len == 0 and T > 1:
                causal = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
                attn_mask = key_mask & causal[None, None, :, :]

                # Left-padding creates early PAD query rows with no valid causal key.
                # Giving each such row access to itself avoids fully-masked SDPA rows
                # on kernels/versions that may otherwise produce NaNs. Real-token
                # attention is unchanged because real positions already include self.
                eye = torch.eye(T, dtype=torch.bool, device=x.device)[None, None, :, :]
                attn_mask = attn_mask | eye
            else:
                attn_mask = key_mask
        elif past_kv is None:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)

        new_kv = (k, v) if use_cache else None
        return y, new_kv


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.ln1(x),
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-style scaled residual projections.
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_ids=None,
        key_valid=None,
    ):
        B, T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        # Tied token/output embedding.
        return F.linear(hidden, self.token_emb.weight)


In [8]:
# CELL 8 — exact supplied Model4 architecture used by the teacher
# CELL 7 — exact Phase-1 Model4 classes (copied from supplied Phase-1 notebook)
# Cell 9 — ~99.31M Efficient Modern Decoder: GQA + RoPE + RMSNorm + SwiGLU   (v2)
# Same parameters / same state_dict keys / same maths as v1 (v1 checkpoints load unchanged). What changed:
#   * RoPE cos/sin are gathered ONCE per forward (v1: once per layer = 34 gathers + 17 GPU->CPU syncs per step)
#   * decode steps use the GQA-native path: the 5 query heads of a KV group become 5 query rows, so the cached
#     K/V are never expanded 5x (v1 copied the whole expanded cache every layer at every generated token)
#   * multi-token chunks that follow a cache are now causal (v1 silently made them bidirectional; unused so far)
#   * build_model() so training always starts from fresh weights; embeddings excluded from weight decay in Cell 11

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32.to(dtype) * self.weight)


class RotaryTable(nn.Module):
    """cos/sin table (non-persistent buffers -> nothing is added to the state_dict)."""
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
        self.max_positions = max_positions

    def forward(self, position_ids):
        # (B,T) -> two (B,1,T,D/2) tensors, shared by every layer
        return (self.cos_cached[position_ids].unsqueeze(1),
                self.sin_cached[position_ids].unsqueeze(1))


def apply_rope(x, cos, sin):
    # x: (B,H,T,D) with interleaved (even, odd) rotation pairs -- identical convention to v1
    B, H, T, D = x.shape
    cos = cos.to(x.dtype)
    sin = sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)


class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH = True     # flip to False to fall back to the (slower) expand-K/V path; used by the self-test

    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        new_kv = (k, v) if use_cache else None          # compact cache: only KV heads are stored
        S = k.size(2)
        drop = self.dropout if self.training else 0.0

        # ---------------- single new token (cached decoding): GQA-native, no K/V expansion
        if T == 1 and past_len > 0 and self.GQA_DECODE_PATH:
            qg = q.reshape(B, KV, self.kv_repeat, D)     # head h = kv*rep + r  ->  row r of group kv
            mask = None if key_valid is None else key_valid[:, None, None, :]
            y = F.scaled_dot_product_attention(qg, k, v, attn_mask=mask, dropout_p=drop, is_causal=False)
            y = y.reshape(B, H, 1, D).transpose(1, 2).contiguous().view(B, 1, C)
            return self.o_proj(y), new_kv

        # ---------------- general path (training / prefill / chunks)
        if self.kv_repeat != 1:
            k_attn = k.repeat_interleave(self.kv_repeat, dim=1)
            v_attn = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            k_attn, v_attn = k, v

        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True                             # fast path used by training
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:                        # left-padding queries keep one safe key (themselves)
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]

        y = F.scaled_dot_product_attention(
            q, k_attn, v_attn, attn_mask=attn_mask, dropout_p=drop, is_causal=is_causal
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.norm1(x), cos, sin, past_kv=past_kv, use_cache=use_cache, key_valid=key_valid
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class EfficientDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
        ffn_dim, dropout, max_positions, rope_base, rms_eps
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(d_model // n_heads, max_positions, rope_base)

        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model, rms_eps)

        self.apply(self._init_weights)
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff.down_proj.weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(self, input_ids, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"context {past_len + T} exceeds max_positions={self.max_positions}")

        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)
        cos, sin = self.rope(position_ids)               # once for all layers (callers keep ids < max_positions)

        x = self.drop(self.token_emb(input_ids))
        new_cache = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, cos, sin, past_kv=past, use_cache=use_cache, key_valid=key_valid)
            if use_cache:
                new_cache.append(kv)
        return self.final_norm(x), new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)   # tied token/output embedding



In [9]:
# CELL 9 — teacher classifier + loader
class TeacherStatuteClassifier(nn.Module):
    def __init__(self, backbone, d_model=640, proj_dim=384, n_labels=100, dropout=0.10):
        super().__init__()
        self.backbone = backbone
        self.proj = nn.Sequential(
            nn.Linear(2*d_model, proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )
        self.chunk_pos = nn.Parameter(torch.zeros(N_CHUNKS, proj_dim))
        nn.init.normal_(self.chunk_pos, std=0.02)
        self.attn = nn.Sequential(
            nn.Linear(proj_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )
        self.head = nn.Sequential(
            nn.Linear(2*proj_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, n_labels),
        )

    def encode_flat_chunks(self, ids, valid):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device)[None, :].expand(B, T)
        cos, sin = self.backbone.rope(pos)
        kv = None   # right-padded causal: valid positions never see pads, identical outputs, faster kernel

        x = self.backbone.drop(self.backbone.token_emb(ids))
        for block in self.backbone.blocks:
            x, _ = block(
                x, cos, sin,
                past_kv=None,
                use_cache=False,
                key_valid=kv,
            )
        x = self.backbone.final_norm(x)

        mask = valid.unsqueeze(-1).to(x.dtype)
        mean = (x * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        last_idx = valid.long().sum(1).clamp_min(1) - 1
        last = x[torch.arange(B, device=x.device), last_idx]
        return self.proj(torch.cat([last, mean], -1))

    def forward(self, input_ids, valid, return_views=False):
        B, C, T = input_ids.shape
        z = self.encode_flat_chunks(
            input_ids.reshape(B*C, T),
            valid.reshape(B*C, T),
        ).reshape(B, C, -1)

        z = z + self.chunk_pos[None, :, :]
        scores = self.attn(z).squeeze(-1)
        weights = torch.softmax(scores, -1)
        attn_pool = (z * weights.unsqueeze(-1)).sum(1)
        max_pool = z.max(1).values
        logits = self.head(torch.cat([attn_pool, max_pool], -1))

        if return_views:
            return logits, weights, z
        return logits, weights

def build_teacher():
    backbone = EfficientDecoderLM(
        vocab_size=len(teacher_tokenizer),
        d_model=M4_D_MODEL,
        n_layers=M4_N_LAYERS,
        n_heads=M4_N_HEADS,
        n_kv_heads=M4_N_KV_HEADS,
        ffn_dim=M4_FFN_DIM,
        dropout=M4_DROPOUT,
        max_positions=M4_MAX_POSITIONS,
        rope_base=M4_ROPE_BASE,
        rms_eps=M4_RMS_EPS,
    )
    return TeacherStatuteClassifier(backbone)

def load_teacher(ckpt_path):
    obj = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False,
    )
    assert isinstance(obj, dict) and "model_state" in obj, obj.keys()
    teacher = build_teacher()
    miss, unexp = teacher.load_state_dict(
        obj["model_state"],
        strict=True,
    )
    assert not miss and not unexp
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    return teacher, obj

class TeacherDataset(Dataset):
    def __init__(self, records, tok, lens):
        self.records = records
        self.tok = tok
        self.lens = lens

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        ids = torch.from_numpy(np.array(self.tok[i], dtype=np.int64, copy=True))
        lens = torch.from_numpy(np.array(self.lens[i], dtype=np.int64, copy=True))
        valid = torch.arange(CHUNK_LEN)[None, :] < lens[:, None]
        return {
            "input_ids": ids,
            "valid": valid,
            "id": str(self.records[i]["id"]),
        }

def make_teacher_loader(records, tok, lens, batch=8):
    return DataLoader(
        TeacherDataset(records, tok, lens),
        batch_size=batch,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
    )

print("Teacher utilities ready.")


Teacher utilities ready.


In [10]:
# CELL 10 — precompute honest TRAIN-only teacher logits + view features
HELDOUT_T_LOGITS = TEACHER_CACHE / "heldout_teacher_train_logits_f16.npy"
HELDOUT_T_VIEWS = TEACHER_CACHE / "heldout_teacher_train_views_f16.npy"
HELDOUT_T_META = TEACHER_CACHE / "heldout_teacher_train_meta.json"

expected_meta = {
    "teacher_sha256": sha256_file(TEACHER_HELDOUT_CKPT),
    "dataset_sha256": DATA_SHA256["train"],
    "n": len(train_records),
    "n_labels": N_LABELS,
    "n_chunks": N_CHUNKS,
    "view_dim": 384,
}

if (
    HELDOUT_T_LOGITS.exists()
    and HELDOUT_T_VIEWS.exists()
    and HELDOUT_T_META.exists()
    and json.load(open(HELDOUT_T_META)) == expected_meta
):
    print("✅ reuse heldout teacher cache")
else:
    teacher, teacher_obj = load_teacher(TEACHER_HELDOUT_CKPT)
    teacher = teacher.to(DEVICE)

    loader = make_teacher_loader(
        train_records,
        t_train_tok,
        t_train_lens,
        batch=16,
    )

    logits_mm = np.lib.format.open_memmap(
        HELDOUT_T_LOGITS,
        mode="w+",
        dtype=np.float16,
        shape=(len(train_records), N_LABELS),
    )
    views_mm = np.lib.format.open_memmap(
        HELDOUT_T_VIEWS,
        mode="w+",
        dtype=np.float16,
        shape=(len(train_records), N_CHUNKS, 384),
    )

    offset = 0
    t0 = time.time()
    with torch.inference_mode():
        for bi, b in enumerate(loader, 1):
            x = b["input_ids"].to(DEVICE, non_blocking=True)
            v = b["valid"].to(DEVICE, non_blocking=True)

            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda" and USE_AMP),
            ):
                lg, _, z = teacher(
                    x, v,
                    return_views=True,
                )

            n = len(b["id"])
            logits_mm[offset:offset+n] = lg.float().cpu().numpy().astype(np.float16)
            views_mm[offset:offset+n] = z.float().cpu().numpy().astype(np.float16)
            offset += n

            if bi % 250 == 0 or bi == len(loader):
                print(
                    f"teacher TRAIN {offset}/{len(train_records)} "
                    f"elapsed={(time.time()-t0)/60:.1f}m",
                    flush=True,
                )

    logits_mm.flush()
    views_mm.flush()
    json.dump(expected_meta, open(HELDOUT_T_META, "w"), indent=2)

    del teacher, logits_mm, views_mm
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

teacher_train_logits = np.load(
    HELDOUT_T_LOGITS,
    mmap_mode="r",
)
teacher_train_views = np.load(
    HELDOUT_T_VIEWS,
    mmap_mode="r",
)

print("teacher logits:", teacher_train_logits.shape)
print("teacher views:", teacher_train_views.shape)


teacher TRAIN 4000/42750 elapsed=1.6m
teacher TRAIN 8000/42750 elapsed=3.4m
teacher TRAIN 12000/42750 elapsed=5.2m
teacher TRAIN 16000/42750 elapsed=7.0m
teacher TRAIN 20000/42750 elapsed=8.8m
teacher TRAIN 24000/42750 elapsed=10.5m
teacher TRAIN 28000/42750 elapsed=12.3m
teacher TRAIN 32000/42750 elapsed=14.1m
teacher TRAIN 36000/42750 elapsed=15.8m
teacher TRAIN 40000/42750 elapsed=17.6m
teacher TRAIN 42750/42750 elapsed=18.8m
teacher logits: (42750, 100)
teacher views: (42750, 4, 384)


In [11]:
# CELL 11 — Model2 student classifier (full transformer fine-tuning)
def build_student_backbone():
    model = DecoderOnlyLM(
        vocab_size=len(student_tokenizer),
        d_model=M2_D_MODEL,
        n_layers=M2_N_LAYERS,
        n_heads=M2_N_HEADS,
        ffn_dim=M2_FFN_DIM,
        dropout=M2_DROPOUT,
        max_positions=M2_MAX_POSITIONS,
    )

    obj = torch.load(
        MODEL2_CKPT,
        map_location="cpu",
        weights_only=False,
    )
    assert isinstance(obj, dict) and "model" in obj
    miss, unexp = model.load_state_dict(
        obj["model"],
        strict=False,
    )
    assert not miss and not unexp, (miss, unexp)

    for p in model.parameters():
        p.requires_grad = True

    if FREEZE_TOKEN_EMBEDDING:
        model.token_emb.weight.requires_grad = False

    return model

class KDStudentClassifier(nn.Module):
    def __init__(
        self,
        backbone,
        d_model=M2_D_MODEL,
        proj_dim=384,
        n_labels=N_LABELS,
        dropout=0.10,
    ):
        super().__init__()
        self.backbone = backbone

        self.proj = nn.Sequential(
            nn.Linear(2*d_model, proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )

        self.chunk_pos = nn.Parameter(
            torch.zeros(N_CHUNKS, proj_dim)
        )
        nn.init.normal_(self.chunk_pos, std=0.02)

        # Label-wise attention gives each statute its own preferred document view.
        self.label_queries = nn.Parameter(
            torch.empty(n_labels, proj_dim)
        )
        self.label_vectors = nn.Parameter(
            torch.empty(n_labels, proj_dim)
        )
        self.label_bias = nn.Parameter(
            torch.zeros(n_labels)
        )
        nn.init.normal_(self.label_queries, std=0.02)
        nn.init.normal_(self.label_vectors, std=0.02)

        # Shared branch preserves broad document/co-occurrence signal.
        self.global_attn = nn.Sequential(
            nn.Linear(proj_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )
        self.global_head = nn.Sequential(
            nn.Linear(2*proj_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, n_labels),
        )

        self.label_scale = nn.Parameter(torch.tensor(1.0))
        self.global_scale = nn.Parameter(torch.tensor(1.0))

    def _block(self, block, x, valid):
        if self.training and USE_GRAD_CHECKPOINTING:
            def fn(h):
                y, _ = block(
                    h,
                    past_kv=None,
                    use_cache=False,
                    key_valid=(None if FAST_CAUSAL_NO_MASK else valid),
                )
                return y
            return checkpoint(
                fn,
                x,
                use_reentrant=False,
            )

        y, _ = block(
            x,
            past_kv=None,
            use_cache=False,
            key_valid=(None if FAST_CAUSAL_NO_MASK else valid),
        )
        return y

    def encode_flat_chunks(self, ids, valid):
        B, T = ids.shape

        position_ids = torch.arange(
            T,
            device=ids.device,
        )[None, :].expand(B, T)

        x = (
            self.backbone.token_emb(ids)
            + self.backbone.pos(position_ids).to(
                self.backbone.token_emb.weight.dtype
            )
        )
        x = self.backbone.drop(x)

        for block in self.backbone.blocks:
            x = self._block(
                block,
                x,
                valid,
            )

        x = self.backbone.final_ln(x)

        mask = valid.unsqueeze(-1).to(x.dtype)
        mean = (
            (x * mask).sum(1)
            / mask.sum(1).clamp_min(1.0)
        )
        last_idx = valid.long().sum(1).clamp_min(1) - 1
        last = x[
            torch.arange(B, device=x.device),
            last_idx,
        ]

        return self.proj(
            torch.cat([last, mean], dim=-1)
        )

    def forward(self, input_ids, valid, return_views=False):
        B, C, T = input_ids.shape

        z = self.encode_flat_chunks(
            input_ids.reshape(B*C, T),
            valid.reshape(B*C, T),
        ).reshape(B, C, -1)

        z = z + self.chunk_pos[None, :, :]

        # Label-wise branch.
        q = F.normalize(
            self.label_queries,
            dim=-1,
        )
        zn = F.normalize(
            z,
            dim=-1,
        )
        scores = torch.einsum(
            "bcd,ld->blc",
            zn,
            q,
        ) * math.sqrt(z.shape[-1])

        label_attn = torch.softmax(
            scores,
            dim=-1,
        )
        label_repr = torch.einsum(
            "blc,bcd->bld",
            label_attn,
            z,
        )
        label_logits = (
            label_repr
            * self.label_vectors[None, :, :]
        ).sum(-1) / math.sqrt(z.shape[-1])
        label_logits = (
            label_logits
            + self.label_bias[None, :]
        )

        # Shared branch.
        gscore = self.global_attn(
            z
        ).squeeze(-1)
        gweight = torch.softmax(
            gscore,
            dim=-1,
        )
        gpool = (
            z * gweight.unsqueeze(-1)
        ).sum(1)
        gmax = z.max(1).values
        glogits = self.global_head(
            torch.cat([gpool, gmax], dim=-1)
        )

        logits = (
            self.label_scale * label_logits
            + self.global_scale * glogits
        )

        diag_attn = (
            label_attn.mean(1)
            + gweight
        ) * 0.5

        if return_views:
            return logits, diag_attn, z
        return logits, diag_attn

student = KDStudentClassifier(
    build_student_backbone()
).to(DEVICE)

total = sum(p.numel() for p in student.parameters())
trainable = sum(
    p.numel()
    for p in student.parameters()
    if p.requires_grad
)

print(
    f"student total={total/1e6:.3f}M "
    f"trainable={trainable/1e6:.3f}M "
    f"({100*trainable/total:.1f}%)"
)


student total=80.690M trainable=64.961M (80.5%)


In [12]:
# CELL 12 — student datasets/loaders with cached teacher targets
class StudentKDDataset(Dataset):
    def __init__(
        self,
        records,
        tok,
        lens,
        teacher_logits=None,
        teacher_views=None,
        labeled=True,
    ):
        self.records = records
        self.tok = tok
        self.lens = lens
        self.teacher_logits = teacher_logits
        self.teacher_views = teacher_views
        self.labeled = labeled

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        ids = torch.from_numpy(
            np.array(
                self.tok[i],
                dtype=np.int64,
                copy=True,
            )
        )
        lens = torch.from_numpy(
            np.array(
                self.lens[i],
                dtype=np.int64,
                copy=True,
            )
        )
        valid = (
            torch.arange(CHUNK_LEN)[None, :]
            < lens[:, None]
        )

        out = {
            "input_ids": ids,
            "valid": valid,
            "id": str(self.records[i]["id"]),
        }

        if self.labeled:
            y = torch.zeros(
                N_LABELS,
                dtype=torch.float32,
            )
            y[self.records[i]["labels"]] = 1.0
            out["labels"] = y

        if self.teacher_logits is not None:
            out["teacher_logits"] = torch.from_numpy(
                np.array(
                    self.teacher_logits[i],
                    dtype=np.float32,
                    copy=True,
                )
            )

        if self.teacher_views is not None:
            out["teacher_views"] = torch.from_numpy(
                np.array(
                    self.teacher_views[i],
                    dtype=np.float32,
                    copy=True,
                )
            )

        return out

def make_student_loader(
    records,
    tok,
    lens,
    batch,
    shuffle,
    labeled=True,
    teacher_logits=None,
    teacher_views=None,
    seed=SEED,
):
    g = torch.Generator()
    g.manual_seed(seed)

    return DataLoader(
        StudentKDDataset(
            records,
            tok,
            lens,
            teacher_logits=teacher_logits,
            teacher_views=teacher_views,
            labeled=labeled,
        ),
        batch_size=batch,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0 and shuffle),
        prefetch_factor=(4 if NUM_WORKERS > 0 else None),
        generator=g,
    )

kd_train_loader = make_student_loader(
    train_records,
    s_train_tok,
    s_train_lens,
    MICRO_BATCH_SIZE,
    True,
    True,
    teacher_train_logits,
    teacher_train_views,
)

sup_train_loader = make_student_loader(
    train_records,
    s_train_tok,
    s_train_lens,
    MICRO_BATCH_SIZE,
    True,
    True,
)

official_loader = make_student_loader(
    dev_raw,
    s_dev_tok,
    s_dev_lens,
    DEV_BATCH_SIZE,
    False,
    True,
)

test_loader = make_student_loader(
    test_records,
    s_test_tok,
    s_test_lens,
    DEV_BATCH_SIZE,
    False,
    False,
)

print(
    "batches:",
    len(kd_train_loader),
    len(official_loader),
    len(test_loader),
)


batches: 10688 637 814


In [13]:
# CELL 13 — exact evaluator logic + threshold tuning
# CELL 10 — exact evaluator logic + threshold tuning
def multilabel_metrics(y_true,y_pred):
    return {
        "macro_f1":float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
        "micro_f1":float(f1_score(y_true,y_pred,average="micro",zero_division=0)),
        "subset_accuracy":float(accuracy_score(y_true,y_pred)),
    }

def apply_thresholds(prob, thresholds, ensure_nonempty=True):
    """Apply per-label thresholds and optionally guarantee >=1 prediction/document."""
    thr=np.asarray(thresholds,dtype=np.float32)
    pred=prob>=thr[None,:]
    if ensure_nonempty:
        empty=np.flatnonzero(~pred.any(1))
        if len(empty):
            # Pick the label closest to clearing its own threshold.
            best=np.argmax(prob[empty]/np.maximum(thr[None,:],1e-6),axis=1)
            pred[empty,best]=True
    return pred

# ---------- v2: vectorised threshold utilities ----------
def macro_f1_fast(y,pred):
    """Identical to sklearn macro-F1 with zero_division=0 (verified in testing), but ~100x faster."""
    yb=y.astype(bool); pb=pred.astype(bool)
    tp=(yb&pb).sum(0).astype(np.float64)
    den=yb.sum(0)+pb.sum(0)
    return float(np.where(den>0,2*tp/np.maximum(den,1),0.0).mean())

def f1_curves(y,prob,grid=THR_GRID):
    """(G,L) matrix: per-label F1 at each grid threshold."""
    yb=y.astype(bool); P=yb.sum(0).astype(np.float64)
    out=np.zeros((len(grid),y.shape[1]),dtype=np.float64)
    for gi,t in enumerate(grid):
        pr=prob>=t
        tp=(pr&yb).sum(0); den=pr.sum(0)+P
        out[gi]=np.where(den>0,2*tp/np.maximum(den,1),0.0)
    return out

def smooth_curves(c):
    """3-point moving average along the threshold axis -> picks the centre of a plateau, not a lucky spike."""
    p=np.pad(c,((1,1),(0,0)),mode="edge")
    return (p[:-2]+p[1:-1]+p[2:])/3.0

def fit_threshold_strategies(y,prob):
    """Return {strategy_name: thresholds(L,)} fitted on (y,prob)."""
    c=f1_curves(y,prob)
    g=float(THR_GRID[int(np.argmax(c.mean(1)))])
    per=THR_GRID[np.argmax(smooth_curves(c),axis=0)].astype(np.float32)
    P=y.sum(0).astype(np.float64)
    out={"global":np.full(y.shape[1],g,dtype=np.float32),"per_label":per}
    for tau in SHRINK_TAUS:
        w=P/(P+tau)
        out[f"shrunk_tau_{int(tau)}"]=(w*per+(1-w)*g).astype(np.float32)
    return out,g,per

def optimize_thresholds(y,prob):
    """Choose the strategy by cross-fitted macro-F1 (fit on half A, score on half B, and vice-versa),
    then refit the winner on ALL given data. Returns the in-sample score too, but selection uses CV."""
    full,g,per=fit_threshold_strategies(y,prob)
    n=len(y); cv={k:[] for k in full}
    for rep_i in range(CV_REPEATS):
        perm=np.random.RandomState(100+rep_i).permutation(n)
        a,b=perm[:n//2],perm[n//2:]
        for tr,te in ((a,b),(b,a)):
            strat,_,_=fit_threshold_strategies(y[tr],prob[tr])
            for k,thr in strat.items():
                cv[k].append(macro_f1_fast(y[te],apply_thresholds(prob[te],thr,ensure_nonempty=True)))
    cv={k:float(np.mean(v)) for k,v in cv.items()}
    name=max(cv,key=cv.get)
    thr=full[name]
    return {"name":name,"thresholds":thr,"cv_macro_f1":cv[name],
            "insample_macro_f1":macro_f1_fast(y,apply_thresholds(prob,thr,ensure_nonempty=True)),
            "cv_all":cv},g,per

@torch.inference_mode()
def predict_loader(model,loader):
    model.eval()
    ids=[]; probs=[]; ys=[]; atts=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda" and USE_AMP)):
            lg,a=model(x,v)
        assert torch.isfinite(lg).all()
        ids.extend(list(b["id"]))
        probs.append(torch.sigmoid(lg.float()).cpu().numpy())
        atts.append(a.float().cpu().numpy())
        if "labels" in b: ys.append(b["labels"].numpy().astype(np.uint8))
    return ids,np.concatenate(probs), (np.concatenate(ys) if ys else None), np.concatenate(atts)

def make_pred_records(ids,prob,thresholds,ensure_nonempty=True):
    binary=apply_thresholds(prob,thresholds,ensure_nonempty=ensure_nonempty)
    return [{"id":str(i),"pred":[int(x) for x in np.flatnonzero(binary[k])]} for k,i in enumerate(ids)]

# Exact supplied evaluator behavior, reproduced in-process.
def exact_phase5_score(truth_records,pred_records):
    t={}; max_label=-1
    for n,r in enumerate(truth_records):
        assert isinstance(r,dict) and "id" in r and "gold" in r
        assert isinstance(r["id"],str) and r["id"].strip() and isinstance(r["gold"],list)
        assert r["id"] not in t
        seen=set()
        for label in r["gold"]:
            assert not isinstance(label,bool) and isinstance(label,int) and label>=0 and label not in seen
            seen.add(label); max_label=max(max_label,label)
        t[r["id"]]=r["gold"]
    nlab=max_label+1
    p={}
    for n,r in enumerate(pred_records):
        assert isinstance(r,dict) and "id" in r and "pred" in r
        assert isinstance(r["id"],str) and isinstance(r["pred"],list) and r["id"] not in p
        seen=set()
        for x in r["pred"]:
            assert not isinstance(x,bool) and isinstance(x,int) and 0<=x<nlab and x not in seen
            seen.add(x)
        p[r["id"]]=r["pred"]
    assert set(t)==set(p)
    yt=np.zeros((len(t),nlab),dtype=np.uint8); yp=np.zeros((len(t),nlab),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    return multilabel_metrics(yt,yp)

print("threshold utilities ready")


threshold utilities ready


In [14]:
# CELL 14 — KD / supervised losses + EMA + optimizer helpers
pos_weight_t = torch.tensor(
    POS_WEIGHT,
    dtype=torch.float32,
    device=DEVICE,
)

def asymmetric_loss(logits, targets):
    xs_pos = torch.sigmoid(logits)
    xs_neg = 1.0 - xs_pos

    if ASL_CLIP > 0:
        xs_neg = (
            xs_neg + ASL_CLIP
        ).clamp(max=1.0)

    loss = (
        targets * torch.log(
            xs_pos.clamp_min(1e-8)
        )
        + (1.0 - targets)
        * torch.log(
            xs_neg.clamp_min(1e-8)
        )
    )

    pt = (
        xs_pos * targets
        + xs_neg * (1.0 - targets)
    )
    gamma = (
        ASL_GAMMA_POS * targets
        + ASL_GAMMA_NEG * (1.0 - targets)
    )

    return -(
        loss
        * torch.pow(
            1.0 - pt,
            gamma,
        )
    ).mean()

def supervised_loss(
    logits,
    targets,
    pos_weight,
):
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight,
    )
    asl = asymmetric_loss(
        logits,
        targets,
    )
    return (
        SUP_BCE_WEIGHT * bce
        + SUP_ASL_WEIGHT * asl
    ), bce.detach(), asl.detach()

def bernoulli_kd_loss(
    student_logits,
    teacher_logits,
    temperature,
):
    """
    Multi-label KD = Bernoulli cross-entropy between
    temperature-softened teacher/student probabilities.
    T^2 restores gradient scale, analogous to standard logit KD.
    """
    T = float(temperature)

    teacher_prob = torch.sigmoid(
        teacher_logits / T
    ).detach()

    loss = F.binary_cross_entropy_with_logits(
        student_logits / T,
        teacher_prob,
    )

    return loss * (T * T)

def feature_kd_loss(
    student_views,
    teacher_views,
):
    # Compare normalized per-view projected representations.
    s = F.normalize(
        student_views.float(),
        dim=-1,
    )
    t = F.normalize(
        teacher_views.float(),
        dim=-1,
    )
    return (
        1.0 - (s * t).sum(-1)
    ).mean()

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.params = {
            n: p
            for n, p in model.named_parameters()
            if p.requires_grad
        }
        self.plist = list(self.params.values())
        self.shadow_list = [p.detach().clone() for p in self.plist]
        self.shadow = dict(zip(self.params.keys(), self.shadow_list))
        self.n = 0

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(self.decay, (1.0 + self.n) / (10.0 + self.n))
        # fused multi-tensor update (identical maths, far fewer kernel launches)
        torch._foreach_mul_(self.shadow_list, d)
        torch._foreach_add_(self.shadow_list, [p.detach() for p in self.plist], alpha=1.0 - d)

    @contextlib.contextmanager
    def applied(self):
        backup = [p.detach().clone() for p in self.plist]
        with torch.no_grad():
            for p, s_ in zip(self.plist, self.shadow_list):
                p.copy_(s_)
        try:
            yield
        finally:
            with torch.no_grad():
                for p, b_ in zip(self.plist, backup):
                    p.copy_(b_)

def get_trainable_state(model):
    return {
        n: p.detach().cpu().clone()
        for n, p in model.named_parameters()
        if p.requires_grad
    }

def load_trainable_state(model, state):
    named = dict(model.named_parameters())
    with torch.no_grad():
        for n, v in state.items():
            named[n].copy_(
                v.to(
                    named[n].device,
                    dtype=named[n].dtype,
                )
            )

def make_llrd_groups(
    model,
    top_lr,
    head_lr,
    decay,
):
    groups = []

    # Full 13-layer student fine-tuning with discriminative LR.
    for bi in range(M2_N_LAYERS):
        depth_from_top = (
            M2_N_LAYERS - 1 - bi
        )
        lr = (
            top_lr
            * (decay ** depth_from_top)
        )

        params = [
            p
            for p in model.backbone.blocks[bi].parameters()
            if p.requires_grad
        ]

        if params:
            groups.append({
                "params": params,
                "lr": lr,
                "weight_decay": WEIGHT_DECAY,
            })

    # Final LN.
    groups.append({
        "params": [
            p
            for p in model.backbone.final_ln.parameters()
            if p.requires_grad
        ],
        "lr": top_lr,
        "weight_decay": 0.0,
    })

    # Optional embedding if unfrozen.
    emb = [
        p
        for p in model.backbone.token_emb.parameters()
        if p.requires_grad
    ]
    if emb:
        groups.append({
            "params": emb,
            "lr": top_lr * 0.25,
            "weight_decay": 0.0,
        })

    head = [
        p
        for n, p in model.named_parameters()
        if p.requires_grad
        and not n.startswith("backbone.")
    ]
    groups.append({
        "params": head,
        "lr": head_lr,
        "weight_decay": WEIGHT_DECAY,
    })

    return groups

def make_optimizer_scheduler(
    model,
    loader,
    epochs,
    top_lr,
    head_lr,
    layer_decay,
):
    groups = make_llrd_groups(
        model,
        top_lr,
        head_lr,
        layer_decay,
    )

    try:
        opt = torch.optim.AdamW(
            groups,
            betas=(0.9, 0.95),
            eps=1e-8,
            fused=(DEVICE.type == "cuda"),
        )
    except Exception:
        opt = torch.optim.AdamW(
            groups,
            betas=(0.9, 0.95),
            eps=1e-8,
        )

    updates_per_epoch = math.ceil(
        len(loader)
        / GRAD_ACCUM_STEPS
    )
    total_updates = (
        updates_per_epoch * epochs
    )
    warmup = max(
        1,
        int(
            WARMUP_RATIO
            * total_updates
        ),
    )

    def lr_lambda(step):
        if step < warmup:
            return (
                step + 1
            ) / warmup

        q = (
            step - warmup
        ) / max(
            1,
            total_updates - warmup,
        )

        return max(
            0.05,
            0.5
            * (
                1.0
                + math.cos(
                    math.pi
                    * min(1.0, q)
                )
            ),
        )

    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lr_lambda,
    )

    return opt, sched


# ---------------- shared epoch runner (same maths as the original inline loops) ----------------
def amp_ctx():
    return torch.amp.autocast(
        "cuda",
        dtype=torch.float16,
        enabled=(DEVICE.type == "cuda" and USE_AMP),
    )

def train_one_epoch(model, loader, opt, sched, scaler, ema, mode, pos_w,
                    tag, epoch, n_epochs, log_every=200):
    """mode='kd'  : soft Bernoulli KD + hard BCE + view-feature KD (Stage A)
       mode='sup' : weighted BCE + ASL on true labels only        (Stage B)
       Running statistics stay on GPU (no per-step host sync)."""
    model.train()
    n = len(loader)
    keys = ["total", "soft", "hard", "feat"] if mode == "kd" else ["total", "bce", "asl"]
    acc = {k: torch.zeros((), device=DEVICE) for k in keys}
    params = [p for p in model.parameters() if p.requires_grad]
    t0 = time.time()

    for bi, b in enumerate(loader, 1):
        x = b["input_ids"].to(DEVICE, non_blocking=True)
        v = b["valid"].to(DEVICE, non_blocking=True)
        y = b["labels"].to(DEVICE, non_blocking=True)

        with amp_ctx():
            if mode == "kd":
                tl = b["teacher_logits"].to(DEVICE, non_blocking=True)
                tv = b["teacher_views"].to(DEVICE, non_blocking=True)
                logits, _, z = model(x, v, return_views=True)
                soft = bernoulli_kd_loss(logits, tl, KD_TEMPERATURE)
                hard = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_w)
                feat = feature_kd_loss(z, tv)
                loss = (
                    KD_SOFT_WEIGHT * soft
                    + KD_HARD_WEIGHT * hard
                    + KD_FEATURE_WEIGHT * feat
                )
                parts = (loss, soft, hard, feat)
            else:
                logits, _ = model(x, v)
                loss, bce, asl = supervised_loss(logits, y, pos_w)
                parts = (loss, bce, asl)

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        for k, p_ in zip(keys, parts):
            acc[k] += p_.detach().float()

        if bi % GRAD_ACCUM_STEPS == 0 or bi == n:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            old_scale = float(scaler.get_scale())
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if float(scaler.get_scale()) >= old_scale:
                sched.step()
                ema.update()

        if bi % log_every == 0 or bi == n:
            vals = {k: float(acc[k]) / bi for k in keys}
            assert all(math.isfinite(x_) for x_ in vals.values()), vals
            el = (time.time() - t0) / 60
            eta = el * (n - bi) / max(1, bi)
            msg = " ".join(f"{k}={vals[k]:.4f}" for k in keys)
            print(f"{tag} epoch={epoch}/{n_epochs} batch={bi}/{n} {msg} "
                  f"elapsed={el:.1f}m ETA={eta:.1f}m", flush=True)
    return {k: float(acc[k]) / n for k in keys}


In [15]:
# CELL 15 — Stage A: teacher-guided distillation
KD_STAGE_CKPT = ROOT / "student_after_kd_stage.pt"

optimizer, scheduler = make_optimizer_scheduler(
    student, kd_train_loader, KD_EPOCHS,
    KD_TOP_BLOCK_LR, KD_HEAD_LR, KD_LAYER_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and USE_AMP))
ema = EMA(student, EMA_DECAY)
optimizer.zero_grad(set_to_none=True)

for epoch in range(1, KD_EPOCHS + 1):
    train_one_epoch(
        student, kd_train_loader, optimizer, scheduler, scaler, ema,
        "kd", pos_weight_t, "KD", epoch, KD_EPOCHS, log_every=200,
    )

    # Diagnostic only (last epoch, cheap global-threshold macro-F1) — Stage A does NOT select the checkpoint.
    if epoch == KD_EPOCHS:
        with ema.applied():
            _, dprob_a, dy_a, _ = predict_loader(student, official_loader)
        c_a = f1_curves(dy_a[CLEAN_IDX], dprob_a[CLEAN_IDX])
        print(f"[KD diagnostic] epoch={epoch} clean global-thr Macro-F1={float(c_a.mean(1).max()):.4f}")

# Save EMA KD-stage initialisation for Stage B.
with ema.applied():
    atomic_torch_save(
        {
            "model_state": student.state_dict(),
            "stage": "after_teacher_guided_kd",
            "kd_epochs": KD_EPOCHS,
            "temperature": KD_TEMPERATURE,
            "soft_weight": KD_SOFT_WEIGHT,
            "hard_weight": KD_HARD_WEIGHT,
            "feature_weight": KD_FEATURE_WEIGHT,
        },
        KD_STAGE_CKPT,
    )

print("✅ Stage A complete:", KD_STAGE_CKPT)

del optimizer, scheduler, scaler, ema
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


KD epoch=1/2 batch=200/10688 total=3.2068 soft=4.1622 hard=0.6996 feat=0.9822 elapsed=0.8m ETA=43.2m
KD epoch=1/2 batch=400/10688 total=2.8890 soft=3.7531 hard=0.5976 feat=0.9856 elapsed=1.6m ETA=41.1m
KD epoch=1/2 batch=600/10688 total=2.7462 soft=3.5658 hard=0.5668 feat=0.9687 elapsed=2.4m ETA=40.2m
KD epoch=1/2 batch=800/10688 total=2.6598 soft=3.4539 hard=0.5449 feat=0.9532 elapsed=3.2m ETA=39.2m
KD epoch=1/2 batch=1000/10688 total=2.5985 soft=3.3776 hard=0.5207 feat=0.9367 elapsed=4.0m ETA=38.4m
KD epoch=1/2 batch=1200/10688 total=2.5519 soft=3.3200 hard=0.5021 feat=0.9202 elapsed=4.8m ETA=37.6m
KD epoch=1/2 batch=1400/10688 total=2.5191 soft=3.2807 hard=0.4864 feat=0.9022 elapsed=5.5m ETA=36.8m
KD epoch=1/2 batch=1600/10688 total=2.4903 soft=3.2466 hard=0.4719 feat=0.8850 elapsed=6.3m ETA=36.0m
KD epoch=1/2 batch=1800/10688 total=2.4655 soft=3.2172 hard=0.4598 feat=0.8681 elapsed=7.1m ETA=35.2m
KD epoch=1/2 batch=2000/10688 total=2.4453 soft=3.1936 hard=0.4491 feat=0.8516 elapsed

In [16]:
# CELL 16 — Stage B: TRUE-LABEL consolidation only + honest DEV selection
# Restore the EMA KD-stage student.
kd_obj = torch.load(KD_STAGE_CKPT, map_location="cpu", weights_only=False)
student.load_state_dict(kd_obj["model_state"], strict=True)
student.to(DEVICE)

optimizer, scheduler = make_optimizer_scheduler(
    student, sup_train_loader, SUP_EPOCHS,
    SUP_TOP_BLOCK_LR, SUP_HEAD_LR, SUP_LAYER_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and USE_AMP))
ema = EMA(student, EMA_DECAY)

BEST_STUDENT = ROOT / "best_kd_student.pt"
best_cv = -1.0
history = []
optimizer.zero_grad(set_to_none=True)

for epoch in range(1, SUP_EPOCHS + 1):
    train_one_epoch(
        student, sup_train_loader, optimizer, scheduler, scaler, ema,
        "sup", pos_weight_t, "SUP", epoch, SUP_EPOCHS, log_every=200,
    )

    with ema.applied():
        dids, dprob, dy, datt = predict_loader(student, official_loader)
        tune, g, per = optimize_thresholds(dy[CLEAN_IDX], dprob[CLEAN_IDX])

        clean_pred = make_pred_records(
            [dids[k] for k in CLEAN_IDX], dprob[CLEAN_IDX], tune["thresholds"], True,
        )
        official_pred = make_pred_records(dids, dprob, tune["thresholds"], True)
        clean_score = exact_phase5_score(clean_truth, clean_pred)
        official_score = exact_phase5_score(official_truth, official_pred)

        row = {
            "sup_epoch": epoch,
            "cv_macro_f1": tune["cv_macro_f1"],
            "threshold_strategy": tune["name"],
            "clean": clean_score,
            "official": official_score,
            "pred_labels": float(np.mean([len(r["pred"]) for r in clean_pred])),
        }
        history.append(row)

        print(
            f"[TRUE LABELS] epoch={epoch} "
            f"CV={tune['cv_macro_f1']:.4f} "
            f"clean={clean_score['macro_f1']:.4f}/"
            f"{clean_score['micro_f1']:.4f}/"
            f"{clean_score['subset_accuracy']:.4f} "
            f"official={official_score['macro_f1']:.4f} "
            f"pred/gold={row['pred_labels']:.2f}/"
            f"{dy[CLEAN_IDX].sum(1).mean():.2f}",
            flush=True,
        )

        if tune["cv_macro_f1"] > best_cv:
            best_cv = tune["cv_macro_f1"]
            atomic_torch_save(
                {
                    "model_state": student.state_dict(),
                    "thresholds": np.asarray(tune["thresholds"], dtype=np.float32),
                    "threshold_strategy": tune["name"],
                    "cv_macro_f1": tune["cv_macro_f1"],
                    "clean_metrics": clean_score,
                    "official_metrics": official_score,
                    "sup_epoch": epoch,
                    "kd_epochs": KD_EPOCHS,
                    "temperature": KD_TEMPERATURE,
                    "teacher_sha256": sha256_file(TEACHER_HELDOUT_CKPT),
                    "student_phase1_sha256": sha256_file(MODEL2_CKPT),
                },
                BEST_STUDENT,
            )
            print("✅ NEW BEST KD STUDENT:", round(best_cv, 5))

json.dump(history, open(ROOT / "history.json", "w"), indent=2)
assert BEST_STUDENT.exists()


SUP epoch=1/2 batch=200/10688 total=0.2277 bce=0.2765 asl=0.0322 elapsed=0.8m ETA=41.5m
SUP epoch=1/2 batch=400/10688 total=0.2229 bce=0.2710 asl=0.0306 elapsed=1.6m ETA=40.7m
SUP epoch=1/2 batch=600/10688 total=0.2234 bce=0.2716 asl=0.0303 elapsed=2.4m ETA=39.9m
SUP epoch=1/2 batch=800/10688 total=0.2224 bce=0.2705 asl=0.0300 elapsed=3.2m ETA=39.1m
SUP epoch=1/2 batch=1000/10688 total=0.2220 bce=0.2701 asl=0.0300 elapsed=4.0m ETA=38.3m
SUP epoch=1/2 batch=1200/10688 total=0.2227 bce=0.2709 asl=0.0300 elapsed=4.8m ETA=37.6m
SUP epoch=1/2 batch=1400/10688 total=0.2244 bce=0.2730 asl=0.0302 elapsed=5.5m ETA=36.8m
SUP epoch=1/2 batch=1600/10688 total=0.2244 bce=0.2730 asl=0.0301 elapsed=6.3m ETA=35.9m
SUP epoch=1/2 batch=1800/10688 total=0.2252 bce=0.2740 asl=0.0302 elapsed=7.1m ETA=35.1m
SUP epoch=1/2 batch=2000/10688 total=0.2259 bce=0.2748 asl=0.0302 elapsed=7.9m ETA=34.3m
SUP epoch=1/2 batch=2200/10688 total=0.2263 bce=0.2754 asl=0.0302 elapsed=8.7m ETA=33.5m
SUP epoch=1/2 batch=2400/

In [17]:
# CELL 17 — reload best heldout KD student + DEV/test probabilities
best = torch.load(
    BEST_STUDENT,
    map_location="cpu",
    weights_only=False,
)

student.load_state_dict(
    best["model_state"],
    strict=True,
)
student.to(DEVICE)
student.eval()

thresholds = np.asarray(
    best["thresholds"],
    dtype=np.float32,
)

print(
    "BEST supervised epoch:",
    best["sup_epoch"],
)
print(
    "CV Macro-F1:",
    best["cv_macro_f1"],
)
print(
    "clean:",
    best["clean_metrics"],
)
print(
    "official:",
    best["official_metrics"],
)
print(
    "threshold strategy:",
    best["threshold_strategy"],
)

dids, dprob, dy, datt = predict_loader(
    student,
    official_loader,
)

tids, tprob, _, tatt = predict_loader(
    student,
    test_loader,
)

np.savez_compressed(
    ROOT / "kd_student_dev_probs.npz",
    id=np.asarray(dids),
    prob=dprob,
    y=dy,
    clean_idx=CLEAN_IDX,
    thresholds=thresholds,
)

np.savez_compressed(
    ROOT / "kd_student_heldout_test_probs.npz",
    id=np.asarray(tids),
    prob=tprob,
    thresholds=thresholds,
)


BEST supervised epoch: 2
CV Macro-F1: 0.3280832120584339
clean: {'macro_f1': 0.33846133999025263, 'micro_f1': 0.4341579874685933, 'subset_accuracy': 0.029315313524793798}
official: {'macro_f1': 0.3381600076168498, 'micro_f1': 0.4336361394278198, 'subset_accuracy': 0.02927020921324035}
threshold strategy: shrunk_tau_25


In [18]:
# CELL 18 — HELDOUT student submission ZIP
heldout_records = make_pred_records(
    tids,
    tprob,
    thresholds,
    True,
)

HELDOUT_PRED = (
    ROOT
    / "prediction_kd_student_heldout.json"
)
json.dump(
    heldout_records,
    open(
        HELDOUT_PRED,
        "w",
        encoding="utf-8",
    ),
    ensure_ascii=False,
)

HELDOUT_ZIP = (
    ROOT
    / "submission_exp6_KD_Model4_to_Model2_HELDOUT.zip"
)
with zipfile.ZipFile(
    HELDOUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    z.write(
        HELDOUT_PRED,
        arcname="prediction.json",
    )

print(
    "HELDOUT labels/doc:",
    np.mean(
        [
            len(r["pred"])
            for r in heldout_records
        ]
    ),
)
print(
    "✅ HELDOUT:",
    HELDOUT_ZIP,
)


HELDOUT labels/doc: 5.618634303709962
✅ HELDOUT: /kaggle/working/phase5_exp6_kd_model4_to_model2_scoremax/submission_exp6_KD_Model4_to_Model2_HELDOUT.zip


## Fresh TRAIN+DEV KD refit

The final student refit starts again from the original Phase-1 Model2 checkpoint.

Teacher:
- `refit_model_selected_weights.pt` from Experiment 1, which legitimately trained on TRAIN+DEV.

Refit sequence:
1. Stage A KD on TRAIN+DEV using the TRAIN+DEV Model4 teacher.
2. Stage B true-label-only consolidation on TRAIN+DEV.
3. Reuse the threshold vector selected by the honest heldout student.

No post-refit DEV calibration is performed.


In [19]:
# CELL 19 — precompute TRAIN+DEV targets from refit Model4 teacher
if not RUN_REFIT:
    print("RUN_REFIT=False -> skip.")
elif not TEACHER_REFIT_CKPT:
    raise FileNotFoundError(
        "RUN_REFIT=True but refit_model_selected_weights.pt is not attached."
    )
else:
    full_records = train_records + dev_raw

    # Tokenize DEV with teacher tokenizer; train already cached.
    # Create one contiguous full teacher cache for simple index alignment.
    t_full_ids_path = CACHE_DIR / "teacher_full_ids.npy"
    t_full_lens_path = CACHE_DIR / "teacher_full_lens.npy"

    if not (
        t_full_ids_path.exists()
        and t_full_lens_path.exists()
    ):
        full_ids = np.concatenate(
            [
                np.asarray(t_train_tok),
                np.asarray(t_dev_tok),
            ],
            axis=0,
        )
        full_lens = np.concatenate(
            [
                np.asarray(t_train_lens),
                np.asarray(t_dev_lens),
            ],
            axis=0,
        )
        np.save(
            t_full_ids_path,
            full_ids,
        )
        np.save(
            t_full_lens_path,
            full_lens,
        )
        del full_ids, full_lens
        gc.collect()

    t_full_tok = np.load(
        t_full_ids_path,
        mmap_mode="r",
    )
    t_full_lens = np.load(
        t_full_lens_path,
        mmap_mode="r",
    )

    REFIT_T_LOGITS = (
        TEACHER_CACHE
        / "refit_teacher_full_logits_f16.npy"
    )
    REFIT_T_VIEWS = (
        TEACHER_CACHE
        / "refit_teacher_full_views_f16.npy"
    )
    REFIT_T_META = (
        TEACHER_CACHE
        / "refit_teacher_full_meta.json"
    )

    expected_refit_meta = {
        "teacher_sha256":
            sha256_file(
                TEACHER_REFIT_CKPT
            ),
        "train_sha256":
            DATA_SHA256["train"],
        "dev_sha256":
            DATA_SHA256["dev"],
        "n":
            len(full_records),
    }

    if (
        REFIT_T_LOGITS.exists()
        and REFIT_T_VIEWS.exists()
        and REFIT_T_META.exists()
        and json.load(
            open(REFIT_T_META)
        ) == expected_refit_meta
    ):
        print(
            "✅ reuse refit teacher cache"
        )
    else:
        refit_teacher, _ = load_teacher(
            TEACHER_REFIT_CKPT
        )
        refit_teacher = refit_teacher.to(
            DEVICE
        )

        loader = make_teacher_loader(
            full_records,
            t_full_tok,
            t_full_lens,
            batch=16,
        )

        lg_mm = np.lib.format.open_memmap(
            REFIT_T_LOGITS,
            mode="w+",
            dtype=np.float16,
            shape=(
                len(full_records),
                N_LABELS,
            ),
        )
        vw_mm = np.lib.format.open_memmap(
            REFIT_T_VIEWS,
            mode="w+",
            dtype=np.float16,
            shape=(
                len(full_records),
                N_CHUNKS,
                384,
            ),
        )

        offset = 0
        t0 = time.time()

        with torch.inference_mode():
            for bi, b in enumerate(
                loader,
                1,
            ):
                x = b[
                    "input_ids"
                ].to(
                    DEVICE,
                    non_blocking=True,
                )
                v = b["valid"].to(
                    DEVICE,
                    non_blocking=True,
                )

                with torch.amp.autocast(
                    "cuda",
                    dtype=torch.float16,
                    enabled=(
                        DEVICE.type
                        == "cuda"
                        and USE_AMP
                    ),
                ):
                    lg, _, z = refit_teacher(
                        x,
                        v,
                        return_views=True,
                    )

                n = len(b["id"])

                lg_mm[
                    offset:offset+n
                ] = (
                    lg.float()
                    .cpu()
                    .numpy()
                    .astype(np.float16)
                )

                vw_mm[
                    offset:offset+n
                ] = (
                    z.float()
                    .cpu()
                    .numpy()
                    .astype(np.float16)
                )

                offset += n

                if (
                    bi % 250 == 0
                    or bi == len(loader)
                ):
                    print(
                        f"refit teacher "
                        f"{offset}/"
                        f"{len(full_records)} "
                        f"elapsed="
                        f"{(time.time()-t0)/60:.1f}m",
                        flush=True,
                    )

        lg_mm.flush()
        vw_mm.flush()

        json.dump(
            expected_refit_meta,
            open(
                REFIT_T_META,
                "w",
            ),
            indent=2,
        )

        del (
            refit_teacher,
            lg_mm,
            vw_mm,
        )
        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    refit_teacher_logits = np.load(
        REFIT_T_LOGITS,
        mmap_mode="r",
    )
    refit_teacher_views = np.load(
        REFIT_T_VIEWS,
        mmap_mode="r",
    )


refit teacher 4000/52931 elapsed=1.8m
refit teacher 8000/52931 elapsed=3.6m
refit teacher 12000/52931 elapsed=5.3m
refit teacher 16000/52931 elapsed=7.1m
refit teacher 20000/52931 elapsed=8.9m
refit teacher 24000/52931 elapsed=10.6m
refit teacher 28000/52931 elapsed=12.4m
refit teacher 32000/52931 elapsed=14.2m
refit teacher 36000/52931 elapsed=15.9m
refit teacher 40000/52931 elapsed=17.7m
refit teacher 44000/52931 elapsed=19.5m
refit teacher 48000/52931 elapsed=21.3m
refit teacher 52000/52931 elapsed=23.1m
refit teacher 52931/52931 elapsed=23.5m


In [20]:
# CELL 20 — fresh Model2 TRAIN+DEV Stage A KD + Stage B true labels
if not RUN_REFIT:
    REFIT_READY = False
    print("RUN_REFIT=False")
else:
    # Free heldout-stage resources first (GPU memory + dataloader workers).
    for _n in ("kd_train_loader", "sup_train_loader"):
        globals().pop(_n, None)
    try:
        student.to("cpu")
    except Exception:
        pass
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    random.seed(REFIT_SEED)
    np.random.seed(REFIT_SEED)
    torch.manual_seed(REFIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(REFIT_SEED)

    full_records = train_records + dev_raw

    # Student full token cache.
    s_full_ids_path = CACHE_DIR / "student_full_ids.npy"
    s_full_lens_path = CACHE_DIR / "student_full_lens.npy"
    if not (s_full_ids_path.exists() and s_full_lens_path.exists()):
        np.save(s_full_ids_path, np.concatenate([np.asarray(s_train_tok), np.asarray(s_dev_tok)], axis=0))
        np.save(s_full_lens_path, np.concatenate([np.asarray(s_train_lens), np.asarray(s_dev_lens)], axis=0))
    s_full_tok = np.load(s_full_ids_path, mmap_mode="r")
    s_full_lens = np.load(s_full_lens_path, mmap_mode="r")

    # Fresh Phase-1 Model2.
    refit_student = KDStudentClassifier(build_student_backbone()).to(DEVICE)

    # Full-data class weights.
    full_counts = np.zeros(N_LABELS, dtype=np.int64)
    for r in full_records:
        full_counts[r["labels"]] += 1
    full_raw_pw = (len(full_records) - full_counts) / full_counts
    full_pos_weight_t = torch.tensor(
        np.clip(np.sqrt(full_raw_pw), 1.0, POS_WEIGHT_CLIP).astype(np.float32),
        dtype=torch.float32, device=DEVICE,
    )

    kd_full_loader = make_student_loader(
        full_records, s_full_tok, s_full_lens, MICRO_BATCH_SIZE, True, True,
        refit_teacher_logits, refit_teacher_views, seed=REFIT_SEED,
    )
    sup_full_loader = make_student_loader(
        full_records, s_full_tok, s_full_lens, MICRO_BATCH_SIZE, True, True,
        seed=REFIT_SEED + 1,
    )

    # ---------------- Refitting Stage A ----------------
    opt, sched = make_optimizer_scheduler(
        refit_student, kd_full_loader, KD_EPOCHS,
        KD_TOP_BLOCK_LR, KD_HEAD_LR, KD_LAYER_DECAY,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and USE_AMP))
    ema = EMA(refit_student, EMA_DECAY)
    opt.zero_grad(set_to_none=True)

    for epoch in range(1, KD_EPOCHS + 1):
        train_one_epoch(
            refit_student, kd_full_loader, opt, sched, scaler, ema,
            "kd", full_pos_weight_t, "REFIT KD", epoch, KD_EPOCHS, log_every=500,
        )

    # Move EMA Stage-A weights into refit student.
    with ema.applied():
        stage_a_state = {k: v.detach().cpu().clone() for k, v in refit_student.state_dict().items()}
    refit_student.load_state_dict(stage_a_state, strict=True)

    del opt, sched, scaler, ema, stage_a_state, kd_full_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    # ---------------- Refitting Stage B ----------------
    # Use the number of supervised epochs selected on honest heldout DEV.
    selected_sup_epochs = int(best["sup_epoch"])

    opt, sched = make_optimizer_scheduler(
        refit_student, sup_full_loader, selected_sup_epochs,
        SUP_TOP_BLOCK_LR, SUP_HEAD_LR, SUP_LAYER_DECAY,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and USE_AMP))
    ema = EMA(refit_student, EMA_DECAY)
    opt.zero_grad(set_to_none=True)

    for epoch in range(1, selected_sup_epochs + 1):
        train_one_epoch(
            refit_student, sup_full_loader, opt, sched, scaler, ema,
            "sup", full_pos_weight_t, "REFIT TRUE", epoch, selected_sup_epochs, log_every=500,
        )

    with ema.applied():
        rids, rprob, _, ratt = predict_loader(refit_student, test_loader)

        REFIT_MODEL = ROOT / "kd_student_refit_selected_weights.pt"
        atomic_torch_save(
            {
                "model_state": {k: v.detach().cpu() for k, v in refit_student.state_dict().items()},
                "thresholds_from_heldout": thresholds,
                "kd_epochs": KD_EPOCHS,
                "supervised_epochs": selected_sup_epochs,
                "teacher_refit_sha256": sha256_file(TEACHER_REFIT_CKPT),
            },
            REFIT_MODEL,
        )

    assert rids == tids
    REFIT_READY = True

    np.savez_compressed(
        ROOT / "kd_student_refit_test_probs.npz",
        id=np.asarray(rids),
        prob=rprob,
        thresholds=thresholds,
    )

    print("✅ KD student refit complete")


REFIT KD epoch=1/2 batch=500/13233 total=2.8649 soft=3.7177 hard=0.6015 feat=0.9974 elapsed=2.0m ETA=50.2m
REFIT KD epoch=1/2 batch=1000/13233 total=2.6476 soft=3.4386 hard=0.5392 feat=0.9567 elapsed=3.9m ETA=48.2m
REFIT KD epoch=1/2 batch=1500/13233 total=2.5432 soft=3.3105 hard=0.4950 feat=0.9155 elapsed=5.9m ETA=46.2m
REFIT KD epoch=1/2 batch=2000/13233 total=2.4813 soft=3.2377 hard=0.4624 feat=0.8769 elapsed=7.9m ETA=44.3m
REFIT KD epoch=1/2 batch=2500/13233 total=2.4394 soft=3.1895 hard=0.4389 feat=0.8407 elapsed=9.9m ETA=42.3m
REFIT KD epoch=1/2 batch=3000/13233 total=2.4097 soft=3.1559 hard=0.4218 feat=0.8086 elapsed=11.8m ETA=40.4m
REFIT KD epoch=1/2 batch=3500/13233 total=2.3871 soft=3.1307 hard=0.4089 feat=0.7807 elapsed=13.8m ETA=38.4m
REFIT KD epoch=1/2 batch=4000/13233 total=2.3699 soft=3.1119 hard=0.3980 feat=0.7568 elapsed=15.8m ETA=36.4m
REFIT KD epoch=1/2 batch=4500/13233 total=2.3570 soft=3.0981 hard=0.3892 feat=0.7361 elapsed=17.7m ETA=34.4m
REFIT KD epoch=1/2 batch=

In [21]:
# CELL 21 — REFIT + HELDOUT/REFIT student ensemble ZIPs
if not RUN_REFIT or not REFIT_READY:
    print("Refit skipped.")
else:
    refit_records = (
        make_pred_records(
            rids,
            rprob,
            thresholds,
            True,
        )
    )

    REFIT_PRED = (
        ROOT
        / "prediction_kd_student_refit.json"
    )

    json.dump(
        refit_records,
        open(
            REFIT_PRED,
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
    )

    REFIT_ZIP = (
        ROOT
        / "submission_exp6_KD_Model4_to_Model2_REFIT.zip"
    )

    with zipfile.ZipFile(
        REFIT_ZIP,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
    ) as z:
        z.write(
            REFIT_PRED,
            arcname=
                "prediction.json",
        )

    ens_prob = (
        (1.0 - REFIT_ENSEMBLE_WEIGHT)
        * tprob
        + REFIT_ENSEMBLE_WEIGHT
        * rprob
    ).astype(np.float32)

    ens_records = (
        make_pred_records(
            tids,
            ens_prob,
            thresholds,
            True,
        )
    )

    ENS_PRED = (
        ROOT
        / "prediction_kd_student_heldout_refit_ensemble.json"
    )

    json.dump(
        ens_records,
        open(
            ENS_PRED,
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
    )

    ENS_ZIP = (
        ROOT
        / "submission_exp6_KD_Model4_to_Model2_HELDOUT_REFIT_ENSEMBLE.zip"
    )

    with zipfile.ZipFile(
        ENS_ZIP,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
    ) as z:
        z.write(
            ENS_PRED,
            arcname=
                "prediction.json",
        )

    np.savez_compressed(
        ROOT
        / "kd_student_heldout_refit_ensemble_test_probs.npz",
        id=np.asarray(tids),
        prob=ens_prob,
        thresholds=thresholds,
        refit_weight=np.float32(
            REFIT_ENSEMBLE_WEIGHT
        ),
    )

    print(
        "✅ REFIT:",
        REFIT_ZIP,
    )
    print(
        "✅ HELDOUT+REFIT:",
        ENS_ZIP,
    )


✅ REFIT: /kaggle/working/phase5_exp6_kd_model4_to_model2_scoremax/submission_exp6_KD_Model4_to_Model2_REFIT.zip
✅ HELDOUT+REFIT: /kaggle/working/phase5_exp6_kd_model4_to_model2_scoremax/submission_exp6_KD_Model4_to_Model2_HELDOUT_REFIT_ENSEMBLE.zip


In [22]:
# CELL 22 (OPTIONAL) — teacher + distilled-student probability blend
# Uses heldout Experiment-1 teacher probabilities if attached.
# Blend weight is selected only on honest clean DEV.
teacher_dev_candidates = [
    p for p in all_files
    if os.path.basename(p) == "dev_official_probs.npz"
]
teacher_test_candidates = [
    p for p in all_files
    if os.path.basename(p) == "heldout_test_probs.npz"
]

if not (
    teacher_dev_candidates
    and teacher_test_candidates
):
    print(
        "Teacher probability files not attached -> blend skipped."
    )
else:
    td = np.load(
        teacher_dev_candidates[0],
        allow_pickle=True,
    )
    tt = np.load(
        teacher_test_candidates[0],
        allow_pickle=True,
    )

    def align(ids_src, prob_src, ids_dst):
        pos = {
            str(x): i
            for i, x
            in enumerate(ids_src)
        }
        return prob_src[
            [
                pos[str(x)]
                for x in ids_dst
            ]
        ]

    teacher_dev_prob = align(
        td["id"],
        td["prob"],
        dids,
    )
    teacher_test_prob = align(
        tt["id"],
        tt["prob"],
        tids,
    )

    blend_rows = []

    for w_student in np.round(
        np.arange(
            0.50,
            1.01,
            0.10,
        ),
        2,
    ):
        pb = (
            w_student * dprob
            + (1.0 - w_student)
            * teacher_dev_prob
        )

        tune_b, _, _ = (
            optimize_thresholds(
                dy[CLEAN_IDX],
                pb[CLEAN_IDX],
            )
        )

        blend_rows.append(
            (
                float(w_student),
                float(
                    tune_b[
                        "cv_macro_f1"
                    ]
                ),
                tune_b,
            )
        )

        print(
            f"student_weight="
            f"{w_student:.2f} "
            f"CV="
            f"{tune_b['cv_macro_f1']:.4f}"
        )

    w_student, _, tune_b = max(
        blend_rows,
        key=lambda x: x[1],
    )

    blend_thr = np.asarray(
        tune_b["thresholds"],
        dtype=np.float32,
    )

    # Honest heldout teacher + honest heldout student.
    blend_test = (
        w_student * tprob
        + (1.0 - w_student)
        * teacher_test_prob
    )

    blend_records = (
        make_pred_records(
            tids,
            blend_test,
            blend_thr,
            True,
        )
    )

    BLEND_ZIP = (
        ROOT
        / "submission_exp6_KD_TEACHER_STUDENT_BLEND.zip"
    )

    blend_json = (
        ROOT
        / "prediction_kd_teacher_student_blend.json"
    )

    json.dump(
        blend_records,
        open(
            blend_json,
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
    )

    with zipfile.ZipFile(
        BLEND_ZIP,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
    ) as z:
        z.write(
            blend_json,
            arcname=
                "prediction.json",
        )

    print(
        "✅ teacher/student blend:",
        BLEND_ZIP,
        "| student weight:",
        w_student,
    )

    # ---- extra variant: heldout+refit students blended with heldout+refit teachers ----
    refit_teacher_test_candidates = [
        p for p in all_files
        if os.path.basename(p) == "refit_test_probs.npz"
    ]
    if RUN_REFIT and REFIT_READY and refit_teacher_test_candidates:
        rt = np.load(refit_teacher_test_candidates[0], allow_pickle=True)
        refit_teacher_test_prob = align(rt["id"], rt["prob"], tids)

        student_mix = (1.0 - REFIT_ENSEMBLE_WEIGHT) * tprob + REFIT_ENSEMBLE_WEIGHT * rprob
        teacher_mix = (1.0 - REFIT_ENSEMBLE_WEIGHT) * teacher_test_prob + REFIT_ENSEMBLE_WEIGHT * refit_teacher_test_prob
        full_blend = (w_student * student_mix + (1.0 - w_student) * teacher_mix).astype(np.float32)

        full_records_blend = make_pred_records(tids, full_blend, blend_thr, True)
        full_json = ROOT / "prediction_kd_full_refit_blend.json"
        json.dump(full_records_blend, open(full_json, "w", encoding="utf-8"), ensure_ascii=False)

        FULL_BLEND_ZIP = ROOT / "submission_exp6_KD_FULL_HELDOUT_REFIT_TEACHER_STUDENT_BLEND.zip"
        with zipfile.ZipFile(FULL_BLEND_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
            z.write(full_json, arcname="prediction.json")
        print("✅ full 4-way blend:", FULL_BLEND_ZIP)
    else:
        print("refit teacher probs (refit_test_probs.npz) not attached or refit skipped -> 4-way blend skipped.")


student_weight=0.50 CV=0.3526
student_weight=0.60 CV=0.3494
student_weight=0.70 CV=0.3452
student_weight=0.80 CV=0.3396
student_weight=0.90 CV=0.3349
student_weight=1.00 CV=0.3281
✅ teacher/student blend: /kaggle/working/phase5_exp6_kd_model4_to_model2_scoremax/submission_exp6_KD_TEACHER_STUDENT_BLEND.zip | student weight: 0.5
✅ full 4-way blend: /kaggle/working/phase5_exp6_kd_model4_to_model2_scoremax/submission_exp6_KD_FULL_HELDOUT_REFIT_TEACHER_STUDENT_BLEND.zip


In [23]:
# CELL 23 — final manifest
manifest = {
    "phase": 5,
    "experiment": 6,
    "version":
        NOTEBOOK_VERSION,
    "method":
        "Knowledge Distillation",
    "teacher":
        "Model4 Partial/Layer-wise FT",
    "student":
        "Phase-1 Model2 decoder-only",
    "distillation": {
        "sequence":
            "teacher-guided Stage A -> true-label-only Stage B",
        "temperature":
            KD_TEMPERATURE,
        "soft_weight":
            KD_SOFT_WEIGHT,
        "hard_weight_stage_a":
            KD_HARD_WEIGHT,
        "feature_weight":
            KD_FEATURE_WEIGHT,
        "stage_b_teacher_weight":
            0.0,
        "kd_epochs":
            KD_EPOCHS,
        "selected_supervised_epochs":
            int(best["sup_epoch"]),
        "teacher_targets":
            "temperature-softened Bernoulli label logits + per-view representation alignment",
    },
    "student_training": {
        "full_transformer_blocks_trainable":
            True,
        "token_embedding_frozen":
            FREEZE_TOKEN_EMBEDDING,
        "gradient_checkpointing":
            USE_GRAD_CHECKPOINTING,
        "views":
            VIEW_NAMES,
        "chunk_len":
            CHUNK_LEN,
        "hard_loss":
            "weighted BCE + asymmetric multilabel loss",
        "ema_decay":
            EMA_DECAY,
    },
    "heldout_selection": {
        "cv_macro_f1":
            best["cv_macro_f1"],
        "clean_metrics":
            best["clean_metrics"],
        "official_metrics":
            best["official_metrics"],
        "threshold_strategy":
            best["threshold_strategy"],
    },
    "assets": {
        "model2_phase1_sha256":
            sha256_file(
                MODEL2_CKPT
            ),
        "teacher_heldout_sha256":
            sha256_file(
                TEACHER_HELDOUT_CKPT
            ),
        "teacher_refit_sha256":
            (
                sha256_file(
                    TEACHER_REFIT_CKPT
                )
                if TEACHER_REFIT_CKPT
                else None
            ),
    },
}

json.dump(
    manifest,
    open(
        ROOT / "manifest.json",
        "w",
    ),
    indent=2,
)

print(
    json.dumps(
        manifest,
        indent=2,
    )
)


{
  "phase": 5,
  "experiment": 6,
  "version": "exp6_model4_to_model2_kd_twostage_fastscore_v3",
  "method": "Knowledge Distillation",
  "teacher": "Model4 Partial/Layer-wise FT",
  "student": "Phase-1 Model2 decoder-only",
  "distillation": {
    "sequence": "teacher-guided Stage A -> true-label-only Stage B",
    "temperature": 2.5,
    "soft_weight": 0.72,
    "hard_weight_stage_a": 0.23,
    "feature_weight": 0.05,
    "stage_b_teacher_weight": 0.0,
    "kd_epochs": 2,
    "selected_supervised_epochs": 2,
    "teacher_targets": "temperature-softened Bernoulli label logits + per-view representation alignment"
  },
  "student_training": {
    "full_transformer_blocks_trainable": true,
    "token_embedding_frozen": true,
    "gradient_checkpointing": true,
    "views": [
      "head",
      "middle",
      "statute_evidence",
      "tail"
    ],
    "chunk_len": 384,
    "hard_loss": "weighted BCE + asymmetric multilabel loss",
    "ema_decay": 0.9985
  },
  "heldout_selection": {
  